# Ingest circuits.csv file
1. Read the file using dataframe reader API
2. Add Metadata Columns
    - Source File
    - Ingestion Timestamp
3. Write to bronze delta table    

##### Step -1 Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField,IntegerType,StringType,DoubleType

circuits_schema = StructType([
    StructField("circuitId",StringType(),True),
    StructField("circuitRef",StringType(),True),
    StructField("name",StringType(),True),
    StructField("country",StringType(),True),
    StructField("lat",DoubleType(),True),
    StructField("lng",DoubleType(),True),
    StructField("url",StringType(),True)
])

In [0]:
circuits_df = ( spark.read.format('csv')
        .option('header','True') 
        # .option('inferSchema','True')\
        .option("mode","FAILFAST")       
        .option("schema",circuits_schema)    
        .load("/Volumes/formula1/landing/files/circuits.csv"))

In [0]:
display(circuits_df)

### Step-2 Add Metadata Columns
 - Ingestion timestamp
 - Source file

In [0]:
from pyspark.sql import functions as F

circuits_final_df = (circuits_df.withColumn("ingestion_timestamp",F.current_timestamp())
                                .withColumn("source_file",F.col('_metadata.file_path')) )

In [0]:
display(circuits_final_df)

#### Step -3 Write to bronze delta table

In [0]:
(
    circuits_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("formula1.bronze.circuits")
)

In [0]:
%sql

SELECT * from formula1.bronze.circuits;